In [58]:
# 1. Standard Data Manipulation
import pandas as pd
import numpy as np
import seaborn as sns

# 2. Data Splitting & Scaling (Crucial for Neural Networks!)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# 3. Handling the Class Imbalance
from imblearn.over_sampling import SMOTE

# 4. Modules for Logistic Regression
from sklearn.ensemble import RandomForestClassifier


In [32]:
df = pd.read_csv('/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.shape

(7043, 21)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [ ]:
df.drop(columns=['customerID'], inplace=True)

df.drop('TotalCharges') # TotalCharges can be found by multiplying tenure and monthly charges

Balance the data


In [34]:
# Convert simple Yes/No to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Quick fix for Gender
df['gender'] = df['gender'].map({'Female': 1, 'Male': 0})

In [35]:
# One-hot encode multi-class variables
nominal_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 
                'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 
                'Contract', 'PaymentMethod']

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True, dtype=int)

In [36]:
# Calculate correlation matrix
corr_matrix = df_encoded.corr()

# Look specifically at what correlates with Churn
print(corr_matrix['Churn'].sort_values(ascending=False))

Churn                                    1.000000
InternetService_Fiber optic              0.307463
PaymentMethod_Electronic check           0.301455
MonthlyCharges                           0.192858
PaperlessBilling                         0.191454
SeniorCitizen                            0.150541
StreamingTV_Yes                          0.063254
StreamingMovies_Yes                      0.060860
MultipleLines_Yes                        0.040033
PhoneService                             0.011691
gender                                   0.008545
MultipleLines_No phone service          -0.011691
DeviceProtection_Yes                    -0.066193
OnlineBackup_Yes                        -0.082307
PaymentMethod_Mailed check              -0.090773
PaymentMethod_Credit card (automatic)   -0.134687
Partner                                 -0.149982
Dependents                              -0.163128
TechSupport_Yes                         -0.164716
OnlineSecurity_Yes                      -0.171270


In [47]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# Check the new balance
print("Original training shape:", y_train.value_counts())
print("Balanced training shape:", y_train_balanced.value_counts())

Original training shape: Churn
0    4130
1    1495
Name: count, dtype: int64
Balanced training shape: Churn
1    4130
0    4130
Name: count, dtype: int64


In [ ]:
# 2. Scale the features (Fit on balanced training data, transform both)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)

In [61]:
lr_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
lr_model.fit(X_train, y_train)



RandomForestClassifier(class_weight='balanced', random_state=42)

In [65]:
y_pred = lr_model.predict(X_test_scaled)

# 2. Print a performance report
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# 3. Print a Confusion Matrix to see exactly where it guessed wrong
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.87      0.74      0.80      1033
           1       0.49      0.70      0.57       374

    accuracy                           0.73      1407
   macro avg       0.68      0.72      0.69      1407
weighted avg       0.77      0.73      0.74      1407

--- Confusion Matrix ---
[[761 272]
 [114 260]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [64]:
import pandas as pd

importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': lr_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importances.head(10))

                           Feature  Importance
8                     TotalCharges    0.176269
4                           tenure    0.167538
7                   MonthlyCharges    0.149750
26               Contract_Two year    0.049771
11     InternetService_Fiber optic    0.044844
28  PaymentMethod_Electronic check    0.036428
25               Contract_One year    0.031874
14              OnlineSecurity_Yes    0.027537
20                 TechSupport_Yes    0.025490
0                           gender    0.025049
